# Checkpoint 4 — Temporal Behavioral Analysis & Inter-Purchase Gaps

## 1. Objective
The objective of Checkpoint 4 is to determine whether customer purchase timelines, empirical inter-purchase intervals, and observation cutoffs provide a reliable, defensible foundation for candidate churn windows ($W \in \{60, 90, 120, 180\}$ days) and subsequent customer segmentation.

Key research questions evaluated:
1. **Customer Purchase Timeline**: What is the overall valid purchase-event timespan, ramp-up, active operating window, and tail-off?
2. **Inter-Purchase Intervals**: What is the empirical distribution of elapsed days between consecutive purchases for repeat customers?
3. **Customer History Depth**: How much historical observation is available for customers prior to each candidate observation cutoff $T_{obs}$?
4. **Candidate Future Windows ($W \in \{60, 90, 120, 180\}$ days)**: What are the return and non-return counts and rates within each window?
5. **Pre-Registered Feasibility Gates**: Which candidate windows satisfy the pre-registered thresholds (`min_positive_cases: 500`, `min_test_positive_cases: 100`)?
6. **Right-Censoring & Post-Cutoff Entrants**: How is right-censoring correctly evaluated for eligible customers, and how many post-cutoff entrants exist?
7. **RFM / Segmentation Implications**: How do inter-purchase dispersion and the ~3% repeat baseline affect conventional clustering vs behavioral segmentation?

> **Diagnostic Policy**: This checkpoint assesses temporal feasibility only. It does not train churn models, run K-Means, or perform train/test splitting.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.temporal_behavior import run_temporal_behavior_audit

print(f"Project root resolved: {project_root}")

## 2. Customer Purchase Timeline & Coverage

Execute the temporal audit module to evaluate overall purchase timespan, monthly volume, and the distinction between dataset coverage and customer observation windows.

In [ ]:
audit_data = run_temporal_behavior_audit(project_root)
tl_data = audit_data["customer_purchase_timeline"]

print(f"Raw data immutability verified: {audit_data['raw_data_immutability_verified']}")
print(f"Earliest Delivered Purchase: {tl_data['earliest_delivered_purchase_timestamp']}")
print(f"Latest Delivered Purchase:   {tl_data['latest_delivered_purchase_timestamp']}")
print(f"Total Timespan:              {tl_data['total_timespan_days']} days ({tl_data['total_active_months']} active months)")
print(f"Peak Month:                  {tl_data['peak_month']} ({tl_data['peak_month_orders']:,} orders)\n")
print(f"Coverage vs Behavioral Window Distinction:\n{tl_data['coverage_vs_behavioral_window_distinction']}\n")

monthly_rows = []
for m, vals in tl_data["monthly_summary"].items():
    monthly_rows.append({
        "Month (YYYY-MM)": m,
        "Delivered Orders": vals["delivered_orders"],
        "Unique Customers": vals["unique_customers"]
    })
pd.DataFrame(monthly_rows)

## 3. Inter-Purchase Interval Analysis for Repeat Customers

Calculate empirical inter-purchase intervals across all consecutive purchase pairs for the repeat customer population (no outliers removed).

In [ ]:
intervals_data = audit_data["inter_purchase_intervals"]
dist = intervals_data["inter_purchase_interval_distribution_days"]
pcts = dist["percentiles"]

df_intervals = pd.DataFrame([
    {"Metric": "Repeat Customers Analyzed", "Value": f"{intervals_data['number_of_repeat_customers']:,}"},
    {"Metric": "Total Purchases among Repeat Customers", "Value": f"{intervals_data['total_purchases_among_repeat_customers']:,}"},
    {"Metric": "Consecutive Purchase Intervals", "Value": f"{intervals_data['number_of_consecutive_purchase_intervals']:,}"},
    {"Metric": "Minimum Interval (Days)", "Value": f"{dist['min']:.4f}"},
    {"Metric": "p01 Interval (Days)", "Value": f"{pcts['p01']:.4f}"},
    {"Metric": "p05 Interval (Days)", "Value": f"{pcts['p05']:.4f}"},
    {"Metric": "p25 Interval (Days)", "Value": f"{pcts['p25']:.4f}"},
    {"Metric": "Median Interval (p50) (Days)", "Value": f"{dist['median']:.4f}"},
    {"Metric": "Mean Interval (Days)", "Value": f"{dist['mean']:.4f}"},
    {"Metric": "p75 Interval (Days)", "Value": f"{pcts['p75']:.4f}"},
    {"Metric": "p90 Interval (Days)", "Value": f"{pcts['p90']:.4f}"},
    {"Metric": "p95 Interval (Days)", "Value": f"{pcts['p95']:.4f}"},
    {"Metric": "p99 Interval (Days)", "Value": f"{pcts['p99']:.4f}"},
    {"Metric": "Maximum Interval (Days)", "Value": f"{dist['max']:.4f}"},
    {"Metric": "Standard Deviation (Days)", "Value": f"{dist['std']:.4f}"}
])

print(f"Behavioral Implication:\n{intervals_data['behavioral_implication_note']}\n")
df_intervals

In [ ]:
tb = intervals_data["interval_threshold_breakdown"]
n_int = intervals_data["number_of_consecutive_purchase_intervals"]

df_buckets = pd.DataFrame([
    {"Interval Horizon": "Same-day / < 1 day", "Interval Count": f"{tb['intervals_under_1_day_same_day']:,}", "Share of Repeat Intervals": f"{tb['intervals_under_1_day_pct']:.2f}%"},
    {"Interval Horizon": "<= 30 days", "Interval Count": f"{tb['intervals_under_30_days']:,}", "Share of Repeat Intervals": f"{tb['intervals_under_30_days_pct']:.2f}%"},
    {"Interval Horizon": "<= 60 days", "Interval Count": f"{tb['intervals_under_60_days']:,}", "Share of Repeat Intervals": f"{tb['intervals_under_60_days_pct']:.2f}%"},
    {"Interval Horizon": "<= 90 days", "Interval Count": f"{tb['intervals_under_90_days']:,}", "Share of Repeat Intervals": f"{tb['intervals_under_90_days_pct']:.2f}%"},
    {"Interval Horizon": "<= 180 days", "Interval Count": f"{tb['intervals_under_180_days']:,}", "Share of Repeat Intervals": f"{tb['intervals_under_180_days_pct']:.2f}%"},
    {"Interval Horizon": "> 365 days", "Interval Count": f"{tb['intervals_over_365_days']:,}", "Share of Repeat Intervals": f"{tb['intervals_over_365_days_pct']:.2f}%"}
])
df_buckets

## 4. Observation Cutoff Feasibility & Non-Circular Window Mechanics

To avoid circularity, observation cutoffs are defined as $T_{obs} = \text{max\_purchase\_ts} - W$. Features are computed strictly using purchases $\le T_{obs}$, while return events are observed strictly in the future window $(T_{obs}, T_{obs} + W]$.

## 5. Customer History Depth Prior to Cutoff

Evaluate the amount of observation history ($T_{obs} - \text{first\_purchase\_timestamp}$) available for eligible customers under each candidate window (descriptive evidence).

In [ ]:
hist_depth_data = audit_data["customer_history_depth"]

depth_rows = []
for w_label, h_info in hist_depth_data.items():
    dist_h = h_info["history_days_distribution"]
    depth_rows.append({
        "Candidate Window": w_label.replace("_", " ").title(),
        "Cutoff T_obs": h_info["observation_cutoff_T_obs"].split("T")[0],
        "Eligible Customers": f"{h_info['eligible_customers_count']:,}",
        "Median History (Days)": f"{dist_h['median']:.1f}",
        "p25 / p75 History (Days)": f"{dist_h['percentiles']['p25']:.1f} / {dist_h['percentiles']['p75']:.1f}",
        ">= 30 Days History (%)": f"{h_info['history_depth_thresholds']['history_ge_30_days']['percentage']:.1f}%",
        ">= 60 Days History (%)": f"{h_info['history_depth_thresholds']['history_ge_60_days']['percentage']:.1f}%",
        ">= 90 Days History (%)": f"{h_info['history_depth_thresholds']['history_ge_90_days']['percentage']:.1f}%",
        ">= 120 Days History (%)": f"{h_info['history_depth_thresholds']['history_ge_120_days']['percentage']:.1f}%",
        ">= 180 Days History (%)": f"{h_info['history_depth_thresholds']['history_ge_180_days']['percentage']:.1f}%",
        ">= 365 Days History (%)": f"{h_info['history_depth_thresholds']['history_ge_365_days']['percentage']:.1f}%"
    })

pd.DataFrame(depth_rows)

## 6. Candidate-Window Comparison & Pre-Registered Feasibility Gates

Compare candidate future windows ($W \in \{60, 90, 120, 180\}$ days) against the pre-registered thresholds (`min_positive_cases: 500`, `min_test_positive_cases: 100`).

In [ ]:
cutoff_data = audit_data["observation_cutoffs"]
w_evals = cutoff_data["candidate_window_evaluations"]

comp_rows = []
for w_key, w_info in w_evals.items():
    gates = w_info["numerical_feasibility_gates"]
    comp_rows.append({
        "Window": f"{w_info['future_window_days']} days",
        "Cutoff T_obs": w_info["observation_cutoff_T_obs"].split("T")[0],
        "Eligible Customers": f"{w_info['eligible_customers']:,}",
        "Returned (Count)": f"{w_info['returned_within_window_count']:,}",
        "Return Rate (%)": f"{w_info['return_rate_pct']:.4f}%",
        "Did Not Return (Count)": f"{w_info['did_not_return_within_window_count']:,}",
        "Non-Return Rate (%)": f"{w_info['non_return_rate_pct']:.4f}%",
        "Projected Test Positives (20%)": f"{w_info['projected_test_positive_cases']:,}",
        "Min Positives Gate (>=500)": "PASS" if gates['min_positive_cases_pass'] else "FAIL",
        "Test Positives Gate (>=100)": "PASS" if gates['min_test_positive_cases_pass'] else "FAIL",
        "Numerical Gates Overall": "PASS" if gates['numerical_gates_overall_pass'] else "FAIL"
    })

print(f"Methodological Framing:\n{cutoff_data['methodological_framing_note']}\n")
pd.DataFrame(comp_rows)

## 7. Right-Censoring & Post-Cutoff Entrant Analysis

Evaluate right-censoring correctly: under $T_{obs} = \text{max\_ts} - W$, every eligible customer has a complete $W$-day future window (right-censored eligible = 0). Customers entering after $T_{obs}$ are tracked as separate excluded cohorts.

In [ ]:
censoring_rows = []
for w_key, w_info in w_evals.items():
    censoring_rows.append({
        "Window": f"{w_info['future_window_days']} days",
        "Cutoff T_obs": w_info["observation_cutoff_T_obs"].split("T")[0],
        "Eligible Population": f"{w_info['eligible_customers']:,} ({w_info['eligible_percentage_of_all_customers']:.2f}%)",
        "Right-Censored Eligible Customers": f"{w_info['right_censored_eligible_customers']} (0.00%)",
        "Post-Cutoff Entrants (Excluded)": f"{w_info['post_cutoff_entrants_count']:,} ({w_info['post_cutoff_entrants_pct']:.2f}%)"
    })

pd.DataFrame(censoring_rows)

## 8. RFM & Customer Segmentation Implications

Assess the empirical impact of the ~3.00% repeat baseline and inter-purchase dispersion on conventional RFM and clustering methodology.

In [ ]:
rfm_imp = audit_data["rfm_segmentation_implications"]

print("=== RFM & SEGMENTATION METHODOLOGICAL ASSESSMENT ===\n")
print(f"1. Repeat Purchase Sparsity:\n   {rfm_imp['repeat_purchase_sparsity_implication']}\n")
print(f"2. Inter-Purchase Interval Dispersion:\n   {rfm_imp['inter_purchase_interval_implication']}\n")
print(f"3. Conventional RFM Utility:\n   {rfm_imp['conventional_rfm_utility']}\n")
print(f"4. Behavioral Segmentation Fallback:\n   {rfm_imp['fallback_status']}\n")

## 9. Temporal Feasibility Decision

Synthesize the measured evidence across candidate windows and analytical domains.

In [ ]:
dec = audit_data["temporal_feasibility_decision"]

decision_rows = [
    {"Evaluation Scope": "60-Day Churn Window", "Status": dec["60_day_churn_window"]["status"], "Evidence / Reason": dec["60_day_churn_window"]["reason"]},
    {"Evaluation Scope": "90-Day Churn Window", "Status": dec["90_day_churn_window"]["status"], "Evidence / Reason": dec["90_day_churn_window"]["reason"]},
    {"Evaluation Scope": "120-Day Churn Window", "Status": dec["120_day_churn_window"]["status"], "Evidence / Reason": dec["120_day_churn_window"]["reason"]},
    {"Evaluation Scope": "180-Day Churn Window", "Status": dec["180_day_churn_window"]["status"], "Evidence / Reason": dec["180_day_churn_window"]["reason"]},
    {"Evaluation Scope": "Preferred Candidate Window", "Status": dec["preferred_candidate_window"], "Evidence / Reason": dec["preferred_candidate_note"]},
    {"Evaluation Scope": "RFM / Behavioral Analysis", "Status": dec["rfm_behavioral_analysis"]["status"], "Evidence / Reason": dec["rfm_behavioral_analysis"]["reason"]},
    {"Evaluation Scope": "Inter-Purchase Analysis", "Status": dec["inter_purchase_analysis"]["status"], "Evidence / Reason": dec["inter_purchase_analysis"]["reason"]},
    {"Evaluation Scope": "Churn Model Case-Count Feasibility", "Status": dec["churn_model_case_count_feasibility"]["status"], "Evidence / Reason": dec["churn_model_case_count_feasibility"]["reason"]},
    {"Evaluation Scope": "Right-Censoring Treatment", "Status": dec["right_censoring_summary"]["status"], "Evidence / Reason": dec["right_censoring_summary"]["reason"]}
]

pd.DataFrame(decision_rows)

## 10. Checkpoint 4 Summary & Next Checkpoint

### Data Analysis Key Findings
- **Inter-Purchase Gaps**: For the 2,801 repeat customers (3,120 intervals), median gap is 29.5 days, mean is 79.2 days, and p75 is 121.5 days. Nearly 25% purchase within < 1 day, while the distribution extends to 609 days.
- **History Depth**: Across all cutoffs, over 87% of eligible customers have $\ge 30$ days of history (median history 136–195 days), confirming sufficient pre-cutoff observation depth.
- **Candidate Window Numerical Gates**: 60-day (282 returns) and 90-day (421 returns) fail the pre-registered 500-positive gate. Both 120-day (539 returns) and 180-day (655 returns) pass the 500-positive and 100-test-positive gates.
- **Right-Censoring**: Correctly defined as 0 among eligible customers under $T_{obs} = \text{max\_ts} - W$. Post-cutoff entrants range from 13.02% (60d) to 40.12% (180d).
- **Raw Data Immutability**: All 9 raw CSV files retained identical SHA-256 hashes pre- and post-validation.

### Insights or Next Steps
- The choice between viable candidate windows (120d vs 180d) and the final framing (churn prediction vs retention analysis fallback) is deferred to the formal Checkpoint 5 methodology evaluation.
- Awaiting user review and authorization before proceeding to Checkpoint 5.